In [ ]:
# Is the NFT clustering repeatable or tied to one seed? Two seeds, synthetic Hamiltonian.

!pip install -q qiskit==1.4.6 qiskit-aer-gpu==0.15.1 qiskit-algorithms==0.4.0 qiskit-optimization==0.7.0 qiskit-ibm-runtime==0.29.0 2>&1 | tail -5

import time, numpy as np
from qiskit.circuit.library import TwoLocal, PauliTwoDesign
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeCairoV2
from qiskit_algorithms import SamplingVQE
from qiskit_algorithms.optimizers import NFT
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

n_qubits = 15
rng_h = np.random.default_rng(0)
terms = []
for i in range(n_qubits):
    lab = ["I"] * n_qubits; lab[i] = "Z"
    terms.append(("".join(lab), float(rng_h.uniform(-2, 2))))
for i in range(n_qubits - 1):
    lab = ["I"] * n_qubits; lab[i] = "Z"; lab[i+1] = "Z"
    terms.append(("".join(lab), float(rng_h.uniform(-2, 2))))
ising_op = SparsePauliOp.from_list(terms)

cairo_nm = NoiseModel.from_backend(FakeCairoV2())
sampler = AerSampler(
    seed=42,
    options={"backend_options": {
        "noise_model": cairo_nm, "method": "statevector", "device": "GPU",
        # NFT evaluates one point at a time, so the batched path shouldn't matter here
        "batched_shots_gpu": False,
        "max_parallel_threads": 0, "max_parallel_experiments": 0,
    }},
)
sampler.options.default_shots = 2000

ansatz_configs = {
    "PauliTwo":          lambda: PauliTwoDesign(num_qubits=n_qubits, reps=3, seed=42),
    "TwoLocal linear":   lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="linear", reps=3),
    "TwoLocal circular": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="circular", reps=3),
    "TwoLocal pairwise": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="pairwise", reps=3),
    "TwoLocal full (control)": lambda: TwoLocal(num_qubits=n_qubits, rotation_blocks=["ry","rz"], entanglement_blocks="cz", entanglement="full", reps=3),
}

results = {}
for seed in [42, 123]:
    print(f"\n{'='*70}\nSEED = {seed}\n{'='*70}")
    for name, factory in ansatz_configs.items():
        ansatz = factory()
        ansatz_d = transpile(ansatz.decompose(reps=10), backend=sampler._backend, optimization_level=1)
        init = np.random.default_rng(seed).uniform(-np.pi, np.pi, size=ansatz_d.num_parameters)
        t0 = time.time()
        vqe = SamplingVQE(sampler=sampler, ansatz=ansatz_d, optimizer=NFT(maxiter=40), initial_point=init)
        result = vqe.compute_minimum_eigenvalue(ising_op)

        bound = ansatz_d.assign_parameters(list(result.optimal_parameters.values())
                                            if isinstance(result.optimal_parameters, dict)
                                            else list(result.optimal_parameters))
        bound.measure_all()
        job = sampler.run([transpile(bound, optimization_level=1)], shots=2000)
        counts = job.result()[0].data.meas.get_counts()
        best_bitstring = max(counts, key=counts.get)

        elapsed = time.time() - t0
        results[(seed, name)] = (float(np.real(result.eigenvalue)), best_bitstring, elapsed)
        print(f"  {name:26s}: energy={float(np.real(result.eigenvalue)):9.4f}  bits={best_bitstring}  ({elapsed:.0f}s)")

print(f"\n--- does the clustering repeat within a seed, and across seeds? ---")
for seed in [42, 123]:
    bitstrings = {name: results[(seed, name)][1] for name in ansatz_configs}
    cluster_group = ["PauliTwo", "TwoLocal linear", "TwoLocal circular", "TwoLocal pairwise"]
    cluster_bits = set(bitstrings[n] for n in cluster_group)
    control_bits = bitstrings["TwoLocal full (control)"]
    print(f"seed={seed}: cluster group all identical? {len(cluster_bits)==1}  "
          f"(distinct bitstrings in group: {len(cluster_bits)})  "
          f"control matches cluster? {control_bits in cluster_bits}")

same_across_seeds = results[(42,"TwoLocal linear")][1] == results[(123,"TwoLocal linear")][1]
print(f"\nsame cluster bitstring across both seeds? {same_across_seeds}")